# 04 · Pianificazione e subagenti

Per compiti complessi servono due idee:
1. un **piano** esplicito che l'agente mantiene mentre lavora;
2. **subagenti**: agenti specializzati con contesto isolato, usati come tool;
3. **routing del modello**: scegliere un modello più forte solo quando serve.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## 1 · Un tool per il piano

Il modo più semplice di dare "pianificazione" è un tool con cui l'agente scrive e aggiorna
una lista di passi. Il piano resta visibile e l'agente può spuntarlo.

In [ ]:
from langchain_core.tools import tool

PIANO: list[str] = []


@tool
def imposta_piano(passi: list[str]) -> str:
    """Definisce l'elenco dei passi da seguire per il compito."""
    PIANO.clear()
    PIANO.extend(passi)
    return "Piano salvato:\n" + "\n".join(f"- {p}" for p in passi)

## 2 · Un subagente usato come tool

Un **subagente** è un secondo agente con un compito ristretto e un proprio contesto.
Lo "impacchettiamo" dentro una funzione-tool: il main agent lo chiama come un tool qualsiasi,
ma il ragionamento del subagente resta separato (non intasa il contesto principale).

In [ ]:
from langchain.agents import create_agent

# Il subagente "ricercatore": un agente semplice con un compito preciso.
ricercatore = create_agent(
    model=model,
    tools=[],
    system_prompt="Sei un ricercatore. Rispondi con 3 punti concisi e concreti.",
)


@tool
def chiedi_al_ricercatore(domanda: str) -> str:
    """Delega una ricerca a un subagente specializzato e restituisce la sintesi."""
    esito = ricercatore.invoke({"messages": [{"role": "user", "content": domanda}]})
    return esito["messages"][-1].text

## 3 · Routing del modello con un middleware

Un **middleware** si interpone attorno alla chiamata del modello. Qui scegliamo un modello
più "forte" quando la richiesta sembra complessa, altrimenti quello economico. Risparmia
soldi e latenza senza rinunciare alla qualità quando serve.

In [ ]:
from langchain.agents.middleware import wrap_model_call

# Modello forte (facoltativo): se non configurato, riusa quello base.
model_forte = ChatOpenAI(model=os.getenv("OPENAI_STRONG_MODEL", MODELLO), use_responses_api=True, store=False)


@wrap_model_call
def instrada_modello(request, handler):
    testo = str(request.messages[-1].content).lower() if request.messages else ""
    # euristica semplice: parole "difficili" -> modello forte
    difficile = any(p in testo for p in ("architettura", "complesso", "approfondito"))
    scelto = model_forte if difficile else model
    return handler(request.override(model=scelto))

## 4 · Il main agent mette tutto insieme

Il coordinatore ha: il tool del piano, il subagente-come-tool e il middleware di routing.

In [ ]:
coordinatore = create_agent(
    model=model,
    tools=[imposta_piano, chiedi_al_ricercatore],
    middleware=[instrada_modello],
    system_prompt=(
        "Per compiti articolati: prima definisci un piano con imposta_piano, "
        "poi usa chiedi_al_ricercatore quando serve approfondire, infine sintetizza."
    ),
)

In [ ]:
esito = coordinatore.invoke({"messages": [{
    "role": "user",
    "content": "Prepara un mini piano per valutare due librerie Python e delega la ricerca.",
}]})
print(esito["messages"][-1].text)
print("\nPiano registrato:", PIANO)

## Prova tu

- Aggiungi un subagente "revisore" che critica il risultato prima della conclusione.
- Stampa `type(m).__name__` sui messaggi per vedere quando parte il subagente.

**Idea chiave**: i subagenti isolano il contesto (meno rumore) e il routing usa il modello
giusto al momento giusto. Sono i mattoni per scalare la complessità.